In [41]:
import pandas as pd
import numpy as np
# sanity check
non_vpn1 = pd.read_pickle("data/nonvpn1_files_AL2.pkl")
non_vpn2 = pd.read_pickle("data/nonvpn2_files_AL2.pkl")
non_vpn3 = pd.read_pickle("data/nonvpn3_files_AL2.pkl")
vpn      = pd.read_pickle("data/vpn_files_AL2.pkl")
tor      = pd.read_pickle("data/tor_files_AL2.pkl")

df_combined = pd.concat([non_vpn1, non_vpn2, non_vpn3, vpn, tor], ignore_index=True)


In [12]:
def get_flow_duration(packet_seq):
    if len(packet_seq) < 2:
        return 0
    timestamps = [p[0] for p in packet_seq]
    return max(timestamps) - min(timestamps)

df_clean = df_combined.copy()
df_clean["flow_duration"] = df_clean["packet_seq"].apply(get_flow_duration)
df_clean["packet_count"] = df_clean["packet_seq"].apply(len)


In [14]:
# cell 3
# zs's code below
#ADJUST THE BUCKETS
#300s coverage: 30 * 10
BUCKET_SIZE_SECONDS = 30
NUM_BUCKETS = 10 

#bytes sent and received per bucket
def bucket_bytes_by_direction(packet_seq, bucket_size=BUCKET_SIZE_SECONDS, num_buckets=NUM_BUCKETS):
    if len(packet_seq) == 0:
        return np.zeros(num_buckets * 2)

    flow_start = packet_seq[0][0]
    bytes_out = np.zeros(num_buckets)
    bytes_in = np.zeros(num_buckets)

    for timestamp, size, direction in packet_seq:
        elapsed = timestamp - flow_start
        bucket_index = int(elapsed // bucket_size)
        if bucket_index < num_buckets:
            if direction > 0:
                bytes_out[bucket_index] += size
            else:
                bytes_in[bucket_index] += size

    return np.concatenate([bytes_out, bytes_in])

#packets sent and received per bucket
def bucket_packets_by_direction(packet_seq, bucket_size=BUCKET_SIZE_SECONDS, num_buckets=NUM_BUCKETS):
    if len(packet_seq) == 0:
        return np.zeros(num_buckets * 2)

    flow_start = packet_seq[0][0]
    packets_out = np.zeros(num_buckets)
    packets_in = np.zeros(num_buckets)

    for timestamp, size, direction in packet_seq:
        elapsed = timestamp - flow_start
        bucket_index = int(elapsed // bucket_size)
        if bucket_index < num_buckets:
            if direction > 0:
                packets_out[bucket_index] += 1
            else:
                packets_in[bucket_index] += 1

    return np.concatenate([packets_out, packets_in])

#mean and std of time between consecutive packets per bucket
def bucket_timing_stats(packet_seq, bucket_size=BUCKET_SIZE_SECONDS, num_buckets=NUM_BUCKETS):
    if len(packet_seq) < 2:
        return np.zeros(num_buckets * 2)  #mean and std per bucket

    flow_start = packet_seq[0][0]
    bucket_gaps = [[] for _ in range(num_buckets)]

    timestamps = [p[0] for p in packet_seq]
    for i in range(1, len(timestamps)):
        gap = timestamps[i] - timestamps[i-1]
        elapsed = timestamps[i] - flow_start
        bucket_index = int(elapsed // bucket_size)
        if bucket_index < num_buckets:
            bucket_gaps[bucket_index].append(gap)

    mean_gaps = np.array([np.mean(g) if g else 0 for g in bucket_gaps])
    std_gaps = np.array([np.std(g) if g else 0 for g in bucket_gaps])

    return np.concatenate([mean_gaps, std_gaps])

#mean and std of bytes of each packet per bucket
def bucket_size_stats(packet_seq, bucket_size=BUCKET_SIZE_SECONDS, num_buckets=NUM_BUCKETS):
    if len(packet_seq) == 0:
        return np.zeros(num_buckets * 2)  #mean and std per bucket

    flow_start = packet_seq[0][0]
    bucket_sizes = [[] for _ in range(num_buckets)]

    for timestamp, size, direction in packet_seq:
        elapsed = timestamp - flow_start
        bucket_index = int(elapsed // bucket_size)
        if bucket_index < num_buckets:
            bucket_sizes[bucket_index].append(size)

    mean_sizes = np.array([np.mean(s) if s else 0 for s in bucket_sizes])
    std_sizes = np.array([np.std(s) if s else 0 for s in bucket_sizes])

    return np.concatenate([mean_sizes, std_sizes])


In [15]:
# another round of zs's code
#8 * 10 more columns here 

#apply and expand into columns: bytes_out_0...9, bytes_in_0...9
bytes_result = df_clean["packet_seq"].apply(bucket_bytes_by_direction)
bytes_cols = [f"bytes_out_{i}" for i in range(NUM_BUCKETS)] + [f"bytes_in_{i}" for i in range(NUM_BUCKETS)]
df_bytes = pd.DataFrame(bytes_result.tolist(), columns=bytes_cols, index=df_clean.index)

#apply and expand into columns: packets_out_0...9, packets_in_0...9
packets_result = df_clean["packet_seq"].apply(bucket_packets_by_direction)
packets_cols = [f"packets_out_{i}" for i in range(NUM_BUCKETS)] + [f"packets_in_{i}" for i in range(NUM_BUCKETS)]
df_packets = pd.DataFrame(packets_result.tolist(), columns=packets_cols, index=df_clean.index)

#timing stats: mean and std of inter-arrival time, per bucket
timing_result = df_clean["packet_seq"].apply(bucket_timing_stats)
timing_cols = [f"gap_mean_{i}" for i in range(NUM_BUCKETS)] + [f"gap_std_{i}" for i in range(NUM_BUCKETS)]
df_timing = pd.DataFrame(timing_result.tolist(), columns=timing_cols, index=df_clean.index)

#size stats: mean and std of packet size, per bucket
size_result = df_clean["packet_seq"].apply(bucket_size_stats)
size_cols = [f"size_mean_{i}" for i in range(NUM_BUCKETS)] + [f"size_std_{i}" for i in range(NUM_BUCKETS)]
df_size = pd.DataFrame(size_result.tolist(), columns=size_cols, index=df_clean.index)

In [16]:
# zs's code AGAIN SORRY
print(f"before shape: {df_clean.shape}")

#combine with original columns
df_clean_features = pd.concat([df_clean, df_bytes, df_packets, df_timing, df_size], axis=1)
print(f"after shape: {df_clean_features.shape}")
df_clean_features.head(3)

before shape: (188670, 10)
after shape: (188670, 90)


,filepath,filename,stream_id,part,network_type,category,flow_key,packet_seq,flow_duration,packet_count,bytes_out_0,bytes_out_1,bytes_out_2,bytes_out_3,bytes_out_4,bytes_out_5,bytes_out_6,bytes_out_7,bytes_out_8,bytes_out_9,bytes_in_0,bytes_in_1,bytes_in_2,bytes_in_3,bytes_in_4,bytes_in_5,bytes_in_6,bytes_in_7,bytes_in_8,bytes_in_9,packets_out_0,packets_out_1,packets_out_2,packets_out_3,packets_out_4,packets_out_5,packets_out_6,packets_out_7,packets_out_8,packets_out_9,packets_in_0,packets_in_1,packets_in_2,packets_in_3,packets_in_4,packets_in_5,packets_in_6,packets_in_7,packets_in_8,packets_in_9,gap_mean_0,gap_mean_1,gap_mean_2,gap_mean_3,gap_mean_4,gap_mean_5,gap_mean_6,gap_mean_7,gap_mean_8,gap_mean_9,gap_std_0,gap_std_1,gap_std_2,gap_std_3,gap_std_4,gap_std_5,gap_std_6,gap_std_7,gap_std_8,gap_std_9,size_mean_0,size_mean_1,size_mean_2,size_mean_3,size_mean_4,size_mean_5,size_mean_6,size_mean_7,size_mean_8,size_mean_9,size_std_0,size_std_1,size_std_2,size_std_3,size_std_4,size_std_5,size_std_6,size_std_7,size_std_8,size_std_9
0,/kaggle/input/datasets/trayne/pcaps-data/NonVP...,email1a.pcap,email1,a,non_vpn,email,"(131.202.240.93, 57806, 255.255.255.255, 10505...","[(1432572763.37583, 127, 1)]",0.000000,1,127.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,127.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/kaggle/input/datasets/trayne/pcaps-data/NonVP...,email1a.pcap,email1,a,non_vpn,email,"(131.202.240.183, 63970, 255.255.255.255, 1900...","[(1432572763.491782, 215, 1), (1432572763.4919...",556.737612,20,430.0,0.0,430.0,0.0,430.0,0.0,430.0,0.0,430.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,2.0,0.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000202,0.0,30.801505,0.0,30.902566,0.0,31.153615,0.0,30.553177,0.0,0.0,0.0,30.801299,0.0,30.902367,0.0,31.153435,0.0,30.552981,0.0,215.0,0.0,215.0,0.0,215.0,0.0,215.0,0.0,215.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/kaggle/input/datasets/trayne/pcaps-data/NonVP...,email1a.pcap,email1,a,non_vpn,email,"(131.202.240.117, 54995, 255.255.255.255, 1050...","[(1432572763.515294, 131, 1)]",0.000000,1,131.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,131.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
feature_cols = bytes_cols + packets_cols + timing_cols + size_cols
X = df_clean_features[feature_cols]
y = df_clean_features["network_type"]

print(X.shape)            # should be (number of flows, 80)
print(y.value_counts())   # see how small Tor really is

(188670, 80)
network_type
non_vpn    169585
vpn         17769
tor          1316
Name: count, dtype: int64


In [18]:
# testing group shuffle split
# from sklearn.model_selection import GroupShuffleSplit

# #use stream_id for grouping the split
# groups = df_clean_features["stream_id"]

# #split 1: 70% train, 30% temp, grouped by stream_id
# gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=4266)
# train_idx, temp_idx = next(gss.split(X, y, groups=groups))

# X_train, X_temp = X.iloc[train_idx], X.iloc[temp_idx]
# y_train, y_temp = y.iloc[train_idx], y.iloc[temp_idx]
# groups_temp = groups.iloc[temp_idx]

# #split 2: temp into 50% validation, 50% test, grouped by stream_id
# gss_temp = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=4266)
# val_idx, test_idx = next(gss_temp.split(X_temp, y_temp, groups=groups_temp))

# X_val, X_test = X_temp.iloc[val_idx], X_temp.iloc[test_idx]
# y_val, y_test = y_temp.iloc[val_idx], y_temp.iloc[test_idx]

# #for verification
# groups_train = groups.iloc[train_idx]
# groups_val = groups_temp.iloc[val_idx]
# groups_test = groups_temp.iloc[test_idx]


In [19]:
# # NOW WITH THE DATA AUGMENTATION FUNCTIONS
# rng = np.random.default_rng(4266)
# WINDOW = BUCKET_SIZE_SECONDS * NUM_BUCKETS   # 300s, same window as the features

# def aug_window_shift(seq, window=WINDOW):
#     """Start the window at a random later point in a long flow (real, unused data)."""
#     t0, t_end = seq[0][0], seq[-1][0]
#     if t_end - t0 <= window:
#         return None
#     start = rng.uniform(t0, t_end - window)
#     sub = [p for p in seq if p[0] >= start]
#     return sub if len(sub) >= 2 else None

# def aug_time_jitter(seq, scale_range=(0.9, 1.1), noise=0.05):
#     """Mildly stretch/shrink the gaps between packets."""
#     times = np.array([p[0] for p in seq])
#     gaps = np.diff(times, prepend=times[0])
#     factor = rng.uniform(*scale_range)
#     gaps = gaps * factor * rng.normal(1, noise, len(gaps)).clip(0.5, 1.5)
#     new_times = times[0] + np.cumsum(gaps)
#     return [(t, s, d) for t, (_, s, d) in zip(new_times, seq)]

# def aug_size_jitter(seq, max_delta=8, min_size=54, max_size=1514):
#     """Nudge packet sizes slightly. NOT used for Tor (fixed 512-byte cells)."""
#     return [(t, int(np.clip(s + rng.integers(-max_delta, max_delta + 1), min_size, max_size)), d)
#             for t, s, d in seq]

# def aug_packet_drop(seq, p=0.03):
#     """Randomly drop ~3% of packets (simulated loss)."""
#     if len(seq) < 5:
#         return list(seq)
#     keep = rng.random(len(seq)) > p
#     keep[0] = True
#     return [pkt for pkt, k in zip(seq, keep) if k]

# def augment_flow(seq, is_tor=False):
#     s = list(seq)
#     if rng.random() < 0.5:
#         shifted = aug_window_shift(s)
#         if shifted:
#             s = shifted
#     s = aug_time_jitter(s)
#     if not is_tor:                    # protect Tor's fixed cell sizes
#         s = aug_size_jitter(s)
#     s = aug_packet_drop(s)
#     return s

# def flow_features(seq):
#     """Recompute the 80 features, same order as X."""
#     return np.concatenate([
#         bucket_bytes_by_direction(seq),
#         bucket_packets_by_direction(seq),
#         bucket_timing_stats(seq),
#         bucket_size_stats(seq),
#     ])

In [20]:
# # AUGMENTING TRAINING SET ONLY
# TARGET_RATIO = 0.5    # grow small classes to 50% of the biggest class...
# MAX_MULTIPLIER = 5    # ...but never more than 5x their original size
# MIN_PACKETS = 5       # only augment flows with enough packets

# train_df = df_clean_features.iloc[train_idx]
# counts = train_df["network_type"].value_counts()
# print("Before:\n", counts, "\n")
# print("Tor streams in train:",
#       train_df[train_df["network_type"].str.lower().str.contains("tor")]["stream_id"].nunique())

# aug_rows, aug_labels = [], []
# for cls, n in counts.items():
#     target = min(int(counts.max() * TARGET_RATIO), n * MAX_MULTIPLIER)
#     n_needed = target - n
#     if n_needed <= 0:
#         continue
#     pool = train_df[(train_df["network_type"] == cls) & (train_df["packet_count"] >= MIN_PACKETS)]
#     if len(pool) == 0:
#         continue
#     is_tor = "tor" in str(cls).lower()
#     sampled = pool.sample(n_needed, replace=True, random_state=4266)
#     for seq in sampled["packet_seq"]:
#         aug_rows.append(flow_features(augment_flow(seq, is_tor=is_tor)))
#         aug_labels.append(cls)

# X_aug = pd.DataFrame(aug_rows, columns=feature_cols)
# y_aug = pd.Series(aug_labels, name="network_type")

# X_train_aug = pd.concat([X_train.reset_index(drop=True), X_aug], ignore_index=True)
# y_train_aug = pd.concat([y_train.reset_index(drop=True), y_aug], ignore_index=True)

# print("\nAfter:\n", y_train_aug.value_counts())

Before:
 network_type
non_vpn    84885
vpn        13203
tor          239
Name: count, dtype: int64 

Tor streams in train: 42

After:
 network_type
non_vpn    84885
vpn        42442
tor         1195
Name: count, dtype: int64


In [21]:
# # FITTING ON TRAIN DATA ONLY
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(np.log1p(X_train_aug))
# X_val_scaled   = scaler.transform(np.log1p(X_val))
# X_test_scaled  = scaler.transform(np.log1p(X_test))


In [22]:
# ## the previous split failed hehe... truying with stratufued group k fold 
# from sklearn.model_selection import StratifiedGroupKFold

# groups = df_clean_features["stream_id"]

# # split 1: ~1/7 (≈14%) test, rest train+val
# sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=4266)
# trainval_idx, test_idx = next(sgkf.split(X, y, groups))

# # split 2: ~1/6 of the remainder (≈14%) val, rest (≈72%) train
# X_tv, y_tv, g_tv = X.iloc[trainval_idx], y.iloc[trainval_idx], groups.iloc[trainval_idx]
# sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=4266)
# tr, va = next(sgkf2.split(X_tv, y_tv, g_tv))

# train_idx = trainval_idx[tr]   # positions in the full dataframe
# val_idx   = trainval_idx[va]

# X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
# X_val,   y_val   = X.iloc[val_idx],   y.iloc[val_idx]
# X_test,  y_test  = X.iloc[test_idx],  y.iloc[test_idx]

# # check: class counts per split
# dist = pd.DataFrame({"train": y_train.value_counts(),
#                      "val": y_val.value_counts(),
#                      "test": y_test.value_counts()})
# dist["% train"] = (dist["train"] / dist.sum(axis=1) * 100).round(1)
# print(dist)

# # check: no stream appears in two splits
# g_tr, g_va, g_te = set(groups.iloc[train_idx]), set(groups.iloc[val_idx]), set(groups.iloc[test_idx])
# print("Overlaps:", len(g_tr & g_va), len(g_tr & g_te), len(g_va & g_te))   # should be 0 0 0

               train    val  test  % train
network_type                              
non_vpn       145500  21329  2756     85.8
vpn            17352    386    31     97.7
tor             1298     12     6     98.6
Overlaps: 0 0 0


In [23]:
# ## cell 9 butb with this extra first line
# AUGMENT_CLASSES = {"tor"}
# TARGET_RATIO = 0.5    # grow small classes to 50% of the biggest class...
# MAX_MULTIPLIER = 5    # ...but never more than 5x their original size
# MIN_PACKETS = 5       # only augment flows with enough packets

# train_df = df_clean_features.iloc[train_idx]
# counts = train_df["network_type"].value_counts()
# print("Before:\n", counts, "\n")
# print("Tor streams in train:",
#       train_df[train_df["network_type"].str.lower().str.contains("tor")]["stream_id"].nunique())

# aug_rows, aug_labels = [], []
# for cls, n in counts.items():
#     target = min(int(counts.max() * TARGET_RATIO), n * MAX_MULTIPLIER)
#     n_needed = target - n
#     if n_needed <= 0:
#         continue
#     pool = train_df[(train_df["network_type"] == cls) & (train_df["packet_count"] >= MIN_PACKETS)]
#     if len(pool) == 0:
#         continue
#     is_tor = "tor" in str(cls).lower()
#     sampled = pool.sample(n_needed, replace=True, random_state=4266)
#     for seq in sampled["packet_seq"]:
#         aug_rows.append(flow_features(augment_flow(seq, is_tor=is_tor)))
#         aug_labels.append(cls)

# X_aug = pd.DataFrame(aug_rows, columns=feature_cols)
# y_aug = pd.Series(aug_labels, name="network_type")

# X_train_aug = pd.concat([X_train.reset_index(drop=True), X_aug], ignore_index=True)
# y_train_aug = pd.concat([y_train.reset_index(drop=True), y_aug], ignore_index=True)

# print("\nAfter:\n", y_train_aug.value_counts())

Before:
 network_type
non_vpn    145500
vpn         17352
tor          1298
Name: count, dtype: int64 

Tor streams in train: 43

After:
 network_type
non_vpn    145500
vpn         72750
tor          6490
Name: count, dtype: int64


In [24]:
# fopr myself to check if the secong plan works
for cls in ["tor", "vpn", "non_vpn"]:
    sizes = (df_clean_features[df_clean_features["network_type"] == cls]
             .groupby("stream_id").size().sort_values(ascending=False))
    print(f"\n=== {cls}: {len(sizes)} streams ===")
    print("Biggest stream's share of class:", round(sizes.iloc[0] / sizes.sum() * 100, 1), "%")
    print(sizes.head(5))


=== tor: 51 streams ===
Biggest stream's share of class: 79.4 %
stream_id
Torrent01      1045
torYoutube2      56
torYoutube1      41
torYoutube3      21
torVimeo1        10
dtype: int64

=== vpn: 23 streams ===
Biggest stream's share of class: 27.8 %
stream_id
vpn_hangouts_audio2    4948
vpn_hangouts_audio1    3172
vpn_hangouts_chat1     2751
vpn_facebook_audio2    1335
vpn_facebook_chat1     1159
dtype: int64

=== non_vpn: 87 streams ===
Biggest stream's share of class: 25.6 %
stream_id
hangouts_audio4    43481
facebook_audio4    42805
skype_file8        30555
skype_audio3       20559
scp1                6846
dtype: int64


In [33]:
# trying a third way: capping flows per stream
stream_info = df_clean_features.groupby(["network_type", "stream_id"]).size()
for cap in [100, 200, 500, 1000]:
    print(cap, stream_info.clip(upper=cap).groupby(level=0).sum().to_dict())

## i cap flows per stream here
MAX_FLOWS_PER_STREAM = 200

# preview class sizes under different caps (useful for the group)
stream_info = df_clean_features.groupby(["network_type", "stream_id"]).size()
print("Flows per class under different caps:")
for cap in [100, 200, 500, 1000, None]:
    counts = stream_info if cap is None else stream_info.clip(upper=cap)
    print(f"  cap={cap}:", counts.groupby(level=0).sum().to_dict())

# shuffle, then keep at most N flows per stream (= random sample per stream)
df_capped = (df_clean_features
             .sample(frac=1, random_state=4266)
             .groupby("stream_id")
             .head(MAX_FLOWS_PER_STREAM)
             .reset_index(drop=True))

print(f"\nUsing cap = {MAX_FLOWS_PER_STREAM}")
print(df_capped["network_type"].value_counts())
tor_streams = df_capped[df_capped["network_type"] == "tor"]["stream_id"].value_counts()
print("Biggest Tor stream's share now:", round(tor_streams.iloc[0] / tor_streams.sum() * 100, 1), "%")


# features + stratified group split here first
from sklearn.model_selection import StratifiedGroupKFold

feature_cols = bytes_cols + packets_cols + timing_cols + size_cols
X = df_capped[feature_cols]
y = df_capped["network_type"]
groups = df_capped["stream_id"]

def make_split(seed):
    sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)
    tv, te = next(sgkf.split(X, y, groups))
    sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)
    tr, va = next(sgkf2.split(X.iloc[tv], y.iloc[tv], groups.iloc[tv]))
    return tv[tr], tv[va], te

def split_error(tr, va, te):
    """Worst-case gap between any class's split % and the 72/14/14 target."""
    pct = pd.DataFrame({"train": y.iloc[tr].value_counts(),
                        "val": y.iloc[va].value_counts(),
                        "test": y.iloc[te].value_counts()}).fillna(0)
    pct = pct.div(pct.sum(axis=1), axis=0)
    target = pd.Series({"train": 0.72, "val": 0.14, "test": 0.14})
    return (pct - target).abs().max().max()

errors = {s: split_error(*make_split(s)) for s in range(100)}
best_seed = min(errors, key=errors.get)
print(f"Best seed: {best_seed} (worst-case gap: {errors[best_seed]:.1%})")

train_idx, val_idx, test_idx = make_split(best_seed)
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val,   y_val   = X.iloc[val_idx],   y.iloc[val_idx]
X_test,  y_test  = X.iloc[test_idx],  y.iloc[test_idx]

# check: counts and % per split
dist = pd.DataFrame({"train": y_train.value_counts(),
                     "val": y_val.value_counts(),
                     "test": y_test.value_counts()}).fillna(0).astype(int)
print(dist, "\n")
print((dist.div(dist.sum(axis=1), axis=0) * 100).round(1), "\n")

# check: no stream in two splits (should be 0 0 0)
g_tr, g_va, g_te = set(groups.iloc[train_idx]), set(groups.iloc[val_idx]), set(groups.iloc[test_idx])
print("Overlaps:", len(g_tr & g_va), len(g_tr & g_te), len(g_va & g_te))

100 {'non_vpn': 6112, 'tor': 371, 'vpn': 1808}
200 {'non_vpn': 9687, 'tor': 471, 'vpn': 3118}
500 {'non_vpn': 15300, 'tor': 771, 'vpn': 6118}
1000 {'non_vpn': 20790, 'tor': 1271, 'vpn': 9404}
Flows per class under different caps:
  cap=100: {'non_vpn': 6112, 'tor': 371, 'vpn': 1808}
  cap=200: {'non_vpn': 9687, 'tor': 471, 'vpn': 3118}
  cap=500: {'non_vpn': 15300, 'tor': 771, 'vpn': 6118}
  cap=1000: {'non_vpn': 20790, 'tor': 1271, 'vpn': 9404}
  cap=None: {'non_vpn': 169585, 'tor': 1316, 'vpn': 17769}

Using cap = 200
network_type
non_vpn    9687
vpn        3118
tor         471
Name: count, dtype: int64
Biggest Tor stream's share now: 42.5 %
Best seed: 2 (worst-case gap: 3.4%)
              train   val  test
network_type                   
non_vpn        6939  1196  1552
vpn            2235   342   541
tor             331    61    79 

              train   val  test
network_type                   
non_vpn        71.6  12.3  16.0
vpn            71.7  11.0  17.4
tor            70.3  1

In [32]:
## augmentation function for the cell above
## this is cell 8
rng = np.random.default_rng(4266)
WINDOW = BUCKET_SIZE_SECONDS * NUM_BUCKETS   # 300s

def aug_window_shift(seq, window=WINDOW):
    t0, t_end = seq[0][0], seq[-1][0]
    if t_end - t0 <= window:
        return None
    start = rng.uniform(t0, t_end - window)
    sub = [p for p in seq if p[0] >= start]
    return sub if len(sub) >= 2 else None

def aug_time_jitter(seq, scale_range=(0.9, 1.1), noise=0.05):
    times = np.array([p[0] for p in seq])
    gaps = np.diff(times, prepend=times[0])
    factor = rng.uniform(*scale_range)
    gaps = gaps * factor * rng.normal(1, noise, len(gaps)).clip(0.5, 1.5)
    new_times = times[0] + np.cumsum(gaps)
    return [(t, s, d) for t, (_, s, d) in zip(new_times, seq)]

def aug_size_jitter(seq, max_delta=8, min_size=54, max_size=1514):
    return [(t, int(np.clip(s + rng.integers(-max_delta, max_delta + 1), min_size, max_size)), d)
            for t, s, d in seq]

def aug_packet_drop(seq, p=0.03):
    if len(seq) < 5:
        return list(seq)
    keep = rng.random(len(seq)) > p
    keep[0] = True
    return [pkt for pkt, k in zip(seq, keep) if k]

def augment_flow(seq, is_tor=False):
    s = list(seq)
    if rng.random() < 0.5:
        shifted = aug_window_shift(s)
        if shifted:
            s = shifted
    s = aug_time_jitter(s)
    if not is_tor:            # protect Tor's fixed 512-byte cell sizes
        s = aug_size_jitter(s)
    s = aug_packet_drop(s)
    return s

def flow_features(seq):
    return np.concatenate([
        bucket_bytes_by_direction(seq),
        bucket_packets_by_direction(seq),
        bucket_timing_stats(seq),
        bucket_size_stats(seq),
    ])

In [34]:
# augment tor in the training set only
# this is cell 9

AUGMENT_CLASSES = {"tor"}   # VPN handled by class weights
TARGET_RATIO = 0.5          # grow to at most 50% of the biggest class...
MAX_MULTIPLIER = 5          # ...and at most 5x original size
MIN_PACKETS = 5

train_df = df_capped.iloc[train_idx]
counts = train_df["network_type"].value_counts()
print("Before:\n", counts, "\n")

aug_rows, aug_labels = [], []
for cls in AUGMENT_CLASSES:
    if cls not in counts:
        continue
    n = counts[cls]
    n_needed = min(int(counts.max() * TARGET_RATIO), n * MAX_MULTIPLIER) - n
    pool = train_df[(train_df["network_type"] == cls) & (train_df["packet_count"] >= MIN_PACKETS)]
    if n_needed <= 0 or len(pool) == 0:
        continue

    print(f"{cls}: {pool['stream_id'].nunique()} streams in pool, "
          f"{(pool['flow_duration'] > WINDOW).mean():.1%} of flows longer than {WINDOW}s")

    # every stream gets an equal chance, so one big stream can't dominate
    stream_sizes = pool["stream_id"].map(pool["stream_id"].value_counts())
    sampled = pool.sample(n_needed, replace=True, weights=1 / stream_sizes, random_state=4266)

    is_tor = "tor" in str(cls).lower()
    for seq in sampled["packet_seq"]:
        aug_rows.append(flow_features(augment_flow(seq, is_tor=is_tor)))
        aug_labels.append(cls)

X_aug = pd.DataFrame(aug_rows, columns=feature_cols)
y_aug = pd.Series(aug_labels, name="network_type")

X_train_aug = pd.concat([X_train.reset_index(drop=True), X_aug], ignore_index=True)
y_train_aug = pd.concat([y_train.reset_index(drop=True), y_aug], ignore_index=True)
print("\nAfter:\n", y_train_aug.value_counts())

Before:
 network_type
non_vpn    6939
vpn        2235
tor         331
Name: count, dtype: int64 

tor: 36 streams in pool, 43.5% of flows longer than 300s

After:
 network_type
non_vpn    6939
vpn        2235
tor        1655
Name: count, dtype: int64


In [35]:
## scale to fit on train only. 
# cell number ten
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(np.log1p(X_train_aug))
X_val_scaled   = scaler.transform(np.log1p(X_val))
X_test_scaled  = scaler.transform(np.log1p(X_test))
print("Train:", X_train_scaled.shape, "| Val:", X_val_scaled.shape, "| Test:", X_test_scaled.shape)

Train: (10829, 80) | Val: (1599, 80) | Test: (2172, 80)


In [37]:
# A/b comparison to ttest if it works
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

def run(Xtr, ytr, name):
    sc = StandardScaler().fit(np.log1p(Xtr))                      # fit on this train set only
    model = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                   random_state=4266, n_jobs=-1)
    model.fit(sc.transform(np.log1p(Xtr)), ytr)
    pred = model.predict(sc.transform(np.log1p(X_val)))
    print(f"\n=== {name} | macro-F1: {f1_score(y_val, pred, average='macro'):.3f} ===")
    print(classification_report(y_val, pred, digits=3))

run(X_train, y_train, "A: no augmentation (class weights only)")
run(X_train_aug, y_train_aug, "B: Tor augmentation + class weights")


=== A: no augmentation (class weights only) | macro-F1: 0.648 ===
              precision    recall  f1-score   support

     non_vpn      0.921     0.912     0.916      1196
         tor      0.130     0.197     0.157        61
         vpn      0.898     0.845     0.870       342

    accuracy                          0.871      1599
   macro avg      0.650     0.651     0.648      1599
weighted avg      0.886     0.871     0.878      1599


=== B: Tor augmentation + class weights | macro-F1: 0.650 ===
              precision    recall  f1-score   support

     non_vpn      0.910     0.909     0.910      1196
         tor      0.160     0.262     0.199        61
         vpn      0.892     0.795     0.841       342

    accuracy                          0.860      1599
   macro avg      0.654     0.655     0.650      1599
weighted avg      0.878     0.860     0.868      1599



In [38]:
# find out why it somehow works but at the expense of VPN data
from sklearn.metrics import confusion_matrix, recall_score

def fit_predict(Xtr, ytr, seed=4266):
    sc = StandardScaler().fit(np.log1p(Xtr))
    m = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=seed, n_jobs=-1)
    m.fit(sc.transform(np.log1p(Xtr)), ytr)
    return m.predict(sc.transform(np.log1p(X_val)))

labels = ["non_vpn", "vpn", "tor"]
versions = {"A (no aug)": (X_train, y_train), "B (aug)": (X_train_aug, y_train_aug)}

# 1. CONFUSION MATRIX: where do Tor flows go, and where did VPN recall go?
for name, (Xtr, ytr) in versions.items():
    cm = confusion_matrix(y_val, fit_predict(Xtr, ytr), labels=labels)
    print(f"\n=== Confusion matrix: {name} (rows = true, cols = predicted) ===")
    print(pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels]))

# 2. WHICH TOR APPS are in train vs val? (tests the 'Torrent dominates' theory)
print("\n=== Tor streams in TRAIN (top 10) ===")
print(df_capped.iloc[train_idx].query("network_type == 'tor'")["stream_id"].value_counts().head(10))

val_df = df_capped.iloc[val_idx].copy()
val_df["pred"] = fit_predict(X_train_aug, y_train_aug)
tor_val = val_df[val_df["network_type"] == "tor"].assign(caught=lambda d: d["pred"].eq("tor"))
print("\n=== Tor streams in VAL: how many flows the model (B) caught ===")
print(tor_val.groupby("stream_id").agg(flows=("caught", "size"), caught=("caught", "sum"))
             .sort_values("flows", ascending=False))

# 3. IS THE TOR GAIN REAL OR LUCK? Repeat with 5 model seeds
rows = []
for seed in range(5):
    for name, (Xtr, ytr) in versions.items():
        pred = fit_predict(Xtr, ytr, seed)
        rows.append({"model": name, "macro_f1": f1_score(y_val, pred, average="macro"),
                     "tor_recall": recall_score(y_val, pred, labels=["tor"], average="macro"),
                     "vpn_recall": recall_score(y_val, pred, labels=["vpn"], average="macro")})
print("\n=== Average over 5 seeds (mean, std) ===")
print(pd.DataFrame(rows).groupby("model").agg(["mean", "std"]).round(3))




=== Confusion matrix: A (no aug) (rows = true, cols = predicted) ===
              pred_non_vpn  pred_vpn  pred_tor
true_non_vpn          1091        28        77
true_vpn                50       289         3
true_tor                44         5        12

=== Confusion matrix: B (aug) (rows = true, cols = predicted) ===
              pred_non_vpn  pred_vpn  pred_tor
true_non_vpn          1087        28        81
true_vpn                67       272         3
true_tor                40         5        16

=== Tor streams in TRAIN (top 10) ===
stream_id
Torrent01                    200
torYoutube3                   21
torVimeo1                     10
tor_spotify2-2                 7
AUDIO_tor_spotify2             7
torVimeo3                      6
torTwitter                     6
BROWSING_tor_browsing_mam      6
torGoogle                      5
BROWSING_tor_browsing_ger      5
Name: count, dtype: int64

=== Tor streams in VAL: how many flows the model (B) caught ===
                 